# 03 - Customers, Sellers, Products and Geography

Olist is a marketplace, so an order links two sides of a two-sided market: a
**customer** somewhere in Brazil and one or more **sellers** somewhere else.
This notebook characterises both sides, the products that move between them,
and the money involved - and ends with the relationship that turns out to
dominate customer satisfaction.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from viz import save_fig, use_report_style

use_report_style()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from data_load import load_all, load_analysis
from features import build_features

In [2]:
tables = load_all()
df = build_features(load_analysis())
print(f"{len(df):,} orders  |  {df.customer_unique_id.nunique():,} distinct customers  "
      f"|  {tables['sellers'].seller_id.nunique():,} sellers  "
      f"|  {tables['products'].product_id.nunique():,} products")

99,091 orders  |  95,773 distinct customers  |  3,095 sellers  |  32,951 products


## 3.1 Geography: a concentrated market

Both sides of the marketplace are concentrated in the south-east, but the
sellers are far more concentrated than the customers.

In [3]:
cust_state = df["customer_state"].value_counts(normalize=True) * 100
sell_state = tables["sellers"]["seller_state"].value_counts(normalize=True) * 100
geo = pd.DataFrame({"customers_%": cust_state, "sellers_%": sell_state}).fillna(0).round(1)
geo["ratio"] = (geo["sellers_%"] / geo["customers_%"].replace(0, np.nan)).round(2)
print(geo.head(10).to_string())

print(f"\nSao Paulo state: {cust_state['SP']:.1f}% of customers but "
      f"{sell_state['SP']:.1f}% of sellers")
print(f"orders shipped within the customer's own state: {100*df['same_state'].mean():.1f}%")
print(f"median customer-seller distance: {df['distance_km'].median():,.0f} km")

    customers_%  sellers_%  ratio
AC          0.1        0.0   0.00
AL          0.4        0.0   0.00
AM          0.1        0.0   0.00
AP          0.1        0.0   0.00
BA          3.4        0.6   0.18
CE          1.3        0.4   0.31
DF          2.2        1.0   0.45
ES          2.0        0.7   0.35
GO          2.0        1.3   0.65
MA          0.7        0.0   0.00

Sao Paulo state: 42.0% of customers but 59.7% of sellers
orders shipped within the customer's own state: 35.7%
median customer-seller distance: 434 km


That asymmetry is the engine of the delivery problem: **59.7% of sellers sit in
São Paulo but only 42% of customers do**, so a large share of orders must cross
the country. Distance is the strongest single correlate of lead time we find.

In [4]:
by_region = df.groupby("customer_region").agg(
    orders=("order_id", "size"),
    median_km=("distance_km", "median"),
    median_days=("target_delivery_days", "median"),
    late_pct=("target_is_late", lambda s: 100 * s.mean()),
).round(1).sort_values("median_days")
print(by_region.to_string())

                 orders  median_km  median_days  late_pct
customer_region                                          
Southeast         68032      329.7          8.7       7.5
South             14091      625.1         12.1       7.1
Central-West       5764      863.3         13.3       8.0
Northeast          9359     1884.7         17.3      14.4
North              1845     2376.3         20.2       9.8


In [5]:
# ---- Figure 4: geography ---------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(config.FIG_WIDTH, 2.6))

top = geo.head(8)
w = 0.4
ix = np.arange(len(top))
axes[0].bar(ix - w/2, top["customers_%"], w, label="customers",
            color=config.PALETTE["primary"])
axes[0].bar(ix + w/2, top["sellers_%"], w, label="sellers",
            color=config.PALETTE["accent"])
axes[0].set_xticks(ix); axes[0].set_xticklabels(top.index, fontsize=7)
axes[0].set_ylabel("% of total")
axes[0].set_title("(a) Sellers are more\nconcentrated than customers", fontsize=8.5)
axes[0].legend(fontsize=7)

axes[1].hist(df["distance_km"].dropna().clip(upper=3000), bins=45,
             color=config.PALETTE["primary"], alpha=0.9)
axes[1].axvline(df["distance_km"].median(), color=config.PALETTE["accent"], lw=1.4)
axes[1].set_xlabel("km (clipped at 3000)")
axes[1].set_title("(b) Customer-seller distance", fontsize=8.5)

r = by_region.sort_values("median_days")
axes[2].barh(r.index, r["median_days"], color=config.PALETTE["primary"], alpha=0.9)
axes[2].set_xlabel("median lead time (days)")
axes[2].set_title("(c) Distance shows up\nin delivery time", fontsize=8.5)
axes[2].tick_params(axis="y", labelsize=7)

fig.tight_layout()
print(save_fig(fig, "fig04_geography"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig04_geography.pdf


## 3.2 Products and categories

In [6]:
cat = df.groupby("lead_category").agg(
    orders=("order_id", "size"),
    median_price=("total_price", "median"),
    median_freight=("total_freight", "median"),
    median_weight_kg=("total_weight_g", lambda s: s.median() / 1000),
    late_pct=("target_is_late", lambda s: 100 * s.mean()),
).sort_values("orders", ascending=False)
print(f"{df['lead_category'].nunique()} categories; top 12 by order count:\n")
print(cat.head(12).round(2).to_string())
print(f"\ntop 10 categories cover {100*cat['orders'].head(10).sum()/len(df):.1f}% of orders")

74 categories; top 12 by order count:

                       orders  median_price  median_freight  median_weight_kg  late_pct
lead_category                                                                          
bed_bath_table           9324         89.00           17.69              1.38      8.81
health_beauty            8757         85.90           16.50              0.45      9.01
sports_leisure           7664         89.99           17.12              0.80      7.80
computers_accessories    6652         86.60           16.79              0.35      7.72
furniture_decor          6279         79.90           19.13              1.80      8.68
housewares               5812         69.80           17.84              1.45      7.04
watches_gifts            5602        139.90           16.14              0.35      8.55
telephony                4169         29.99           15.10              0.25      8.56
auto                     3869         90.00           18.23              0.95    

## 3.3 Money: price, freight and instalments

Freight is a large fraction of order value, and instalment payment is the norm
rather than the exception - both are Brazilian e-commerce characteristics that
matter for any commercial framing.

In [7]:
print(f"median order value    : R$ {df['total_price'].median():,.2f}")
print(f"median freight        : R$ {df['total_freight'].median():,.2f}")
print(f"median freight ratio  : {100*df['freight_ratio'].median():.1f}% of item value")
print(f"\npayment type mix (%):")
print((df['lead_payment_type'].value_counts(normalize=True) * 100).round(1).to_string())
print(f"\norders paid in more than one instalment: {100*(df['max_installments']>1).mean():.1f}%")
print(f"median instalments when > 1           : {df.loc[df['max_installments']>1,'max_installments'].median():.0f}")

median order value    : R$ 86.90
median freight        : R$ 17.17
median freight ratio  : 22.4% of item value

payment type mix (%):
lead_payment_type
credit_card    75.4
boleto         19.9
voucher         3.2
debit_card      1.5
not_defined     0.0

orders paid in more than one instalment: 51.4%
median instalments when > 1           : 4


In [8]:
# ---- Figure 5: categories and money ---------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(config.FIG_WIDTH, 2.6))

t = cat.head(10).sort_values("orders")
axes[0].barh(range(len(t)), t["orders"], color=config.PALETTE["primary"], alpha=0.9)
axes[0].set_yticks(range(len(t)))
axes[0].set_yticklabels([s.replace("_", " ")[:22] for s in t.index], fontsize=6.5)
axes[0].set_xlabel("orders")
axes[0].set_title("(a) Top product categories", fontsize=8.5)

axes[1].hist(df["total_price"].clip(upper=600), bins=45,
             color=config.PALETTE["gold"], alpha=0.9)
axes[1].axvline(df["total_price"].median(), color=config.PALETTE["accent"], lw=1.4)
axes[1].set_xlabel("R$ (clipped at 600)")
axes[1].set_title("(b) Order value is\nright-skewed", fontsize=8.5)

pm = (df["lead_payment_type"].value_counts(normalize=True) * 100).head(4)
axes[2].bar(range(len(pm)), pm.values, color=config.PALETTE["green"], alpha=0.9)
axes[2].set_xticks(range(len(pm)))
axes[2].set_xticklabels([s.replace("_", "\n") for s in pm.index], fontsize=7)
axes[2].set_ylabel("% of orders")
axes[2].set_title("(c) Payment method", fontsize=8.5)

fig.tight_layout()
print(save_fig(fig, "fig05_products_money"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig05_products_money.pdf


## 3.4 Reviews, and the relationship that matters most

Review scores are heavily skewed towards 5 stars - the class-imbalance problem
the brief warns about. But the interesting structure is *what* drives a bad
review.

In [9]:
rs = tables["order_reviews"]["review_score"].value_counts(normalize=True).sort_index() * 100
print("review score distribution (%):")
print(rs.round(2).to_string())
print(f"\n4-5 stars: {rs[4]+rs[5]:.1f}%   1-2 stars: {rs[1]+rs[2]:.1f}%")
print(f"orders with a review: {100*df['review_score'].notna().mean():.1f}%")

review score distribution (%):
review_score
1    11.51
2     3.18
3     8.24
4    19.29
5    57.78

4-5 stars: 77.1%   1-2 stars: 14.7%
orders with a review: 99.2%


In [10]:
r = df.dropna(subset=["review_score", "target_is_late"])
print("mean review score by delivery outcome:")
print(r.groupby(r["target_is_late"].map({0: "on time", 1: "late"}))["review_score"]
      .agg(["mean", "median", "size"]).round(2).to_string())

low_late = 100 * r.loc[r["target_is_late"] == 1, "target_low_review"].mean()
low_ok = 100 * r.loc[r["target_is_late"] == 0, "target_low_review"].mean()
print(f"\n1-2 star rate when delivered LATE    : {low_late:.1f}%")
print(f"1-2 star rate when delivered ON TIME : {low_ok:.1f}%")
print(f"multiplier                           : {low_late/low_ok:.1f}x")

# and as a smooth function of how late
r2 = r.dropna(subset=["delay_vs_estimate_days"]).copy()
bins = [-np.inf, -20, -10, -5, 0, 5, 10, 20, np.inf]
labels = ["20+ early", "10-20 early", "5-10 early", "0-5 early",
          "0-5 late", "5-10 late", "10-20 late", "20+ late"]
r2["bucket"] = pd.cut(r2["delay_vs_estimate_days"], bins=bins, labels=labels)
prof = r2.groupby("bucket", observed=True).agg(
    mean_score=("review_score", "mean"),
    low_pct=("target_low_review", lambda s: 100 * s.mean()),
    n=("order_id", "size"),
).round(2)
print("\nreview quality against delivery earliness/lateness:")
print(prof.to_string())

mean review score by delivery outcome:
                mean  median   size
target_is_late                     
late            2.57     2.0   7659
on time         4.30     5.0  87902

1-2 star rate when delivered LATE    : 54.0%
1-2 star rate when delivered ON TIME : 9.2%
multiplier                           : 5.9x

review quality against delivery earliness/lateness:
             mean_score  low_pct      n
bucket                                 
20+ early          4.29    10.17  13170
10-20 early        4.33     8.56  43482
5-10 early         4.28     9.14  22437
0-5 early          4.15    11.05   8813
0-5 late           3.46    27.98   3567
5-10 late          1.89    73.78   1861
10-20 late         1.67    80.17   1382
20+ late           1.76    77.39    849


This is the strongest relationship in the dataset. Being delivered late raises
the probability of a 1-2 star review roughly **six-fold**, and the effect is
monotone in how late the parcel is. It is the reason Candidate 2 in the next
notebook is framed *after* delivery rather than at checkout.

In [11]:
# ---- Figure 6: reviews and delivery ---------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(config.FIG_WIDTH, 2.7))

axes[0].bar(rs.index, rs.values, color=config.PALETTE["primary"], alpha=0.9)
axes[0].set_xlabel("review score"); axes[0].set_ylabel("% of reviews")
axes[0].set_title("(a) Reviews skew hard to 5 stars", fontsize=9)
for i, v in rs.items():
    axes[0].text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=7)

ax = axes[1]
ax.bar(range(len(prof)), prof["low_pct"], color=config.PALETTE["accent"], alpha=0.9)
ax.set_xticks(range(len(prof)))
ax.set_xticklabels(prof.index, rotation=45, ha="right", fontsize=6.5)
ax.set_ylabel("% scoring 1-2 stars")
ax.axvline(3.5, color="black", ls="--", lw=1)
ax.annotate("promised date", xy=(3.5, ax.get_ylim()[1]*0.9), xytext=(4, 0),
            textcoords="offset points", fontsize=7)
ax.set_title("(b) Lateness drives dissatisfaction", fontsize=9)

fig.tight_layout()
print(save_fig(fig, "fig06_reviews_delivery"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig06_reviews_delivery.pdf


## 3.5 Correlations

In [12]:
num = df[["target_delivery_days", "estimated_days", "distance_km", "total_freight",
          "total_price", "total_weight_g", "n_items", "review_score"]].rename(columns={
    "target_delivery_days": "lead time", "estimated_days": "promised days",
    "distance_km": "distance", "total_freight": "freight", "total_price": "price",
    "total_weight_g": "weight", "n_items": "items", "review_score": "review"})
corr = num.corr()
print(corr.round(3).to_string())

               lead time  promised days  distance  freight  price  weight  items  review
lead time          1.000          0.384     0.395    0.167  0.055   0.071 -0.019  -0.334
promised days      0.384          1.000     0.523    0.242  0.076   0.077  0.015  -0.052
distance           0.395          0.523     1.000    0.313  0.079  -0.010 -0.017  -0.059
freight            0.167          0.242     0.313    1.000  0.413   0.639  0.437  -0.089
price              0.055          0.076     0.079    0.413  1.000   0.349  0.153  -0.040
weight             0.071          0.077    -0.010    0.639  0.349   1.000  0.228  -0.055
items             -0.019          0.015    -0.017    0.437  0.153   0.228  1.000  -0.115
review            -0.334         -0.052    -0.059   -0.089 -0.040  -0.055 -0.115   1.000


In [13]:
# ---- Figure 7: correlation matrix -----------------------------------------
fig, ax = plt.subplots(figsize=(config.FIG_WIDTH * 0.62, 3.1))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45,
                                                    ha="right", fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(v) > 0.55 else "black")
ax.grid(False)
ax.set_title("Correlations among order-level quantities")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
fig.tight_layout()
print(save_fig(fig, "fig07_correlations"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig07_correlations.pdf


## 3.6 Takeaways

* The marketplace is **geographically lopsided**: 59.7% of sellers are in São
  Paulo against 42% of customers, so most orders travel. Median customer-seller
  distance is around 434 km, and distance is the strongest checkout-time
  correlate of lead time (r = 0.40).
* Product demand is **long-tailed** - the top ten of 74 categories cover roughly
  two-thirds of orders.
* **Freight is not a rounding error**: the median order pays freight worth about
  a fifth of item value, and three-quarters of orders are paid by credit card,
  usually in instalments.
* Reviews are **heavily imbalanced** (77% four or five stars), and the dominant
  driver of a bad review is a late delivery: 1-2 star rate rises from 9.2% to
  54.0% when an order misses its promised date, monotonically in how late it is.

In [14]:
assert 100*df["same_state"].mean() < 50
assert low_late > 4 * low_ok, "lateness should dominate dissatisfaction"
assert rs[4] + rs[5] > 70, "reviews should be strongly imbalanced"
print("Entity assertions passed.")

Entity assertions passed.
